In [ ]:
pip install tensorflow==2.13.0

ERROR: Could not find a version that satisfies the requirement tensorflow==2.13.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.13.0


In [ ]:
"""
train_lstm_burnout.py  —  TF 2.16+ / Keras 3.x  (Colab compatible)
====================================================================
Trains on Colab (TF 2.16+) and saves in a format your backend can load.

Saves:
  burnout_lstm_model.h5       ← load on backend with tf.keras
  lstm_scaler.pkl
  feature_cols.pkl
  baseline_stats.pkl
  app_category_weights.json
"""

DATA_PATH  = None
EPOCHS     = 150
BATCH_SIZE = 64

import os, json
import numpy as np
import pandas as pd
import joblib

import tensorflow as tf
# Force Keras 2 behaviour so .h5 is compatible with backend TF 2.16
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tf_keras as keras                  # tf_keras = Keras 2 bundled with TF 2.16+
from tf_keras import layers
from tf_keras import callbacks as cb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print(f"TensorFlow {tf.__version__}  |  tf_keras {keras.__version__}")

# ─── App Category Weights ─────────────────────────────────────────────────────
APP_CATEGORY_WEIGHTS = {
    "instagram":  0.90, "tiktok":     0.95, "facebook":   0.85,
    "twitter":    0.80, "snapchat":   0.85, "reddit":     0.70,
    "whatsapp":   0.50, "telegram":   0.45, "discord":    0.50,
    "youtube":    0.60, "netflix":    0.65,
    "news":       0.55, "inshorts":   0.55,
    "gmail":     -0.20, "outlook":   -0.20, "slack":     -0.15,
    "zoom":      -0.10, "teams":     -0.10, "notion":    -0.30,
    "docs":      -0.25, "sheets":    -0.25, "calendar":  -0.30,
    "github":    -0.35, "vscode":    -0.40, "jira":      -0.25,
    "figma":     -0.20, "trello":    -0.25,
    "headspace": -0.50, "calm":      -0.50,
    "strava":    -0.40, "fitbit":    -0.30,
    "__unknown__": 0.30,
}

# ─── Features ─────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    "screen_time_hours", "app_switches_per_hour", "unique_apps_per_day",
    "social_app_ratio", "work_app_ratio", "entertainment_ratio",
    "wellness_ratio", "weighted_app_burnout", "sleep_hours", "sleep_quality",
    "exercise_min_per_week", "social_hours_per_week", "call_count",
    "missed_call_ratio", "sms_count",
    "screen_time_delta", "app_switches_delta", "social_ratio_delta",
    "work_ratio_delta", "sleep_delta",
    "screen_time_zscore", "app_switches_zscore", "social_ratio_zscore",
]

TARGET_COL = "burnout_score"
SEQ_LEN    = 2
N_FEATURES = len(FEATURE_COLS)   # 23


# ─── Custom Attention layer (no Lambda — fully serialisable) ──────────────────

class AttentionPool(keras.layers.Layer):
    """
    Weighted temporal pooling over LSTM output.
    No Lambda layer so .h5 serialises cleanly across TF versions.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = keras.layers.Dense(1, activation="tanh")

    def call(self, inputs):
        # inputs: (B, seq_len, hidden)
        scores = self.score_dense(inputs)          # (B, seq_len, 1)
        scores = tf.nn.softmax(scores, axis=1)     # (B, seq_len, 1)
        return tf.reduce_sum(inputs * scores, axis=1)  # (B, hidden)

    def get_config(self):
        cfg = super().get_config()
        return cfg


# ─── Model ────────────────────────────────────────────────────────────────────

def build_model(seq_len, n_features):
    inp = keras.Input(shape=(seq_len, n_features), name="sequence_input")

    x = layers.LSTM(64, return_sequences=True,
                    kernel_regularizer=keras.regularizers.l2(1e-4))(inp)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(32, return_sequences=True,
                    kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.2)(x)

    x = AttentionPool(name="attention_pool")(x)    # (B, 32)

    x   = layers.Dense(32, activation="relu")(x)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid", name="burnout_score")(x)

    model = keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=["mae", keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model


# ─── Feature Engineering ──────────────────────────────────────────────────────

def compute_weighted_app_burnout(row):
    score = (
          row.get("social_app_ratio",    0) * 0.85
        + row.get("entertainment_ratio", 0) * 0.60
        - row.get("work_app_ratio",      0) * 0.25
        - row.get("wellness_ratio",      0) * 0.40
    )
    return float(np.clip(score, -1.0, 1.0))


def add_deviation_features(df, baseline):
    mapping = {
        "screen_time_hours":     ("screen_time_delta",  "screen_time_zscore"),
        "app_switches_per_hour": ("app_switches_delta", "app_switches_zscore"),
        "social_app_ratio":      ("social_ratio_delta", "social_ratio_zscore"),
        "work_app_ratio":        ("work_ratio_delta",   None),
        "sleep_hours":           ("sleep_delta",        None),
    }
    for col, (delta_col, z_col) in mapping.items():
        mean, std = baseline[col]
        df[delta_col] = df[col] - mean
        if z_col:
            df[z_col] = df[delta_col] / std
    return df


def compute_burnout_labels(df):
    s = np.zeros(len(df))
    s += np.clip(df["screen_time_hours"] / 14, 0, 1)        * 0.15
    s += df["social_app_ratio"].clip(0, 1)                   * 0.20
    s += (1 - df["work_app_ratio"].clip(0, 1))               * 0.10
    s += df["entertainment_ratio"].clip(0, 1)                * 0.08
    s += np.clip(df["app_switches_per_hour"] / 80, 0, 1)    * 0.12
    s += (1 - np.clip(df["sleep_hours"] / 9, 0, 1))         * 0.10
    s += (1 - (df["sleep_quality"] - 1) / 4)                * 0.07
    s += (1 - np.clip(df["exercise_min_per_week"]/300,0,1)) * 0.05
    s += df["missed_call_ratio"].clip(0, 1)                  * 0.03
    s += df["weighted_app_burnout"].clip(-1, 1)              * 0.10
    dev = (
          df["screen_time_zscore"].clip(-3, 3)  * 0.08
        + df["social_ratio_zscore"].clip(-3, 3) * 0.10
        - df["app_switches_zscore"].clip(-3, 3) * 0.02
    ) / 3
    s += dev.clip(-0.10, 0.30)
    return np.clip(s, 0.0, 1.0).astype(np.float32)


# ─── Synthetic Data ───────────────────────────────────────────────────────────

def generate_synthetic_data(n_users=600, days_per_user=30):
    rng = np.random.default_rng(42)
    records = []
    for uid in range(n_users):
        base_screen   = rng.uniform(3.0, 8.0)
        base_social   = rng.uniform(0.1, 0.5)
        base_work     = rng.uniform(0.2, 0.6)
        base_switches = rng.uniform(10,  40)
        base_sleep    = rng.uniform(5.5, 8.5)

        for day in range(days_per_user):
            spike = rng.random() < 0.25
            if spike:
                screen  = min(base_screen + rng.uniform(2, 5), 16)
                social  = min(base_social + rng.uniform(0.2, 0.4), 1.0)
                work    = max(base_work   - rng.uniform(0.1, 0.3), 0.0)
                sw      = base_switches   + rng.uniform(15, 40)
                sleep   = max(base_sleep  - rng.uniform(0.5, 2.0), 2.0)
            else:
                screen  = max(base_screen + rng.normal(0, 0.5), 1.0)
                social  = float(np.clip(base_social + rng.normal(0, 0.05), 0, 1))
                work    = float(np.clip(base_work   + rng.normal(0, 0.05), 0, 1))
                sw      = max(base_switches + rng.normal(0, 5), 2.0)
                sleep   = float(np.clip(base_sleep  + rng.normal(0, 0.4), 2, 10))

            entertain = float(np.clip(rng.uniform(0, 0.3) + (0.15 if spike else 0), 0, 1))
            wellness  = float(np.clip(rng.uniform(0, 0.2) - (0.05 if spike else 0), 0, 1))
            total = social + work + entertain + wellness
            if total > 1.0:
                f = 1.0 / total
                social *= f; work *= f; entertain *= f; wellness *= f

            sq = int(np.clip(int(rng.integers(1, 6)) - (2 if spike else 0), 1, 5))
            records.append({
                "user_id": uid, "day": day,
                "screen_time_hours":     round(float(screen), 2),
                "app_switches_per_hour": round(float(sw), 1),
                "unique_apps_per_day":   int(rng.integers(5, 40)),
                "social_app_ratio":      round(social, 3),
                "work_app_ratio":        round(work, 3),
                "entertainment_ratio":   round(entertain, 3),
                "wellness_ratio":        round(wellness, 3),
                "sleep_hours":           round(float(sleep), 2),
                "sleep_quality":         sq,
                "exercise_min_per_week": round(float(max(0, rng.uniform(0,400)-(100 if spike else 0))), 0),
                "social_hours_per_week": round(float(rng.uniform(2, 30)), 1),
                "call_count":            int(rng.integers(0, 20)),
                "missed_call_ratio":     round(float(np.clip(rng.uniform(0,0.4)+(0.2 if spike else 0),0,1)), 2),
                "sms_count":             int(rng.integers(0, 80)),
            })
    return pd.DataFrame(records)


# ─── Sequence Builder ─────────────────────────────────────────────────────────

def build_sequences(df, scaler, baseline, fit_scaler=True):
    df = df.copy()
    df = add_deviation_features(df, baseline)
    df["weighted_app_burnout"] = df.apply(compute_weighted_app_burnout, axis=1)
    df[TARGET_COL]             = compute_burnout_labels(df)

    all_X, all_y = [], []
    for _, group in df.groupby("user_id"):
        group = group.sort_values("day").reset_index(drop=True)
        if len(group) < SEQ_LEN:
            continue
        feats  = group[FEATURE_COLS].values.astype(np.float32)
        labels = group[TARGET_COL].values.astype(np.float32)
        for i in range(len(group) - SEQ_LEN + 1):
            all_X.append(feats[i : i + SEQ_LEN])
            all_y.append(labels[i + SEQ_LEN - 1])

    X = np.array(all_X, dtype=np.float32)
    y = np.array(all_y, dtype=np.float32)
    N, S, F = X.shape
    X_flat  = X.reshape(N * S, F)
    X_flat  = scaler.fit_transform(X_flat) if fit_scaler else scaler.transform(X_flat)
    return X_flat.reshape(N, S, F), y


# ─── RUN ──────────────────────────────────────────────────────────────────────

def run(data_path=None):

    # 1. Data
    if data_path and os.path.exists(data_path):
        print(f"Loading: {data_path}")
        df = pd.read_csv(data_path)
        for col in ["social_app_ratio","work_app_ratio","entertainment_ratio","wellness_ratio"]:
            if col not in df.columns: df[col] = 0.0
        if "user_id" not in df.columns: df["user_id"] = 0
        if "day"     not in df.columns: df["day"]     = range(len(df))
    else:
        print("Generating synthetic data …")
        df = generate_synthetic_data(600, 30)
        df.to_csv("synthetic_burnout_data.csv", index=False)
        print(f"Saved synthetic_burnout_data.csv  ({len(df)} rows)")

    print(f"Dataset: {len(df)} rows | {df['user_id'].nunique()} users\n")

    # 2. Baseline
    key_sigs = ["screen_time_hours","app_switches_per_hour",
                "social_app_ratio","work_app_ratio","sleep_hours"]
    baseline = {c: (float(df[c].mean()), float(max(df[c].std(), 1e-6))) for c in key_sigs}
    print("Baseline (mean ± std):")
    for k,(m,s) in baseline.items():
        print(f"  {k:<32}  {m:.3f} ± {s:.3f}")

    # 3. Sequences
    scaler = StandardScaler()
    X, y   = build_sequences(df, scaler, baseline, fit_scaler=True)
    print(f"\nSequences: X={X.shape}  y={y.shape}")
    print(f"Burnout  min={y.min():.3f}  mean={y.mean():.3f}  max={y.max():.3f}")

    # 4. Split
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.20, random_state=42)
    X_v,  X_te,  y_v,  y_te  = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42)
    print(f"Train {len(X_tr)} | Val {len(X_v)} | Test {len(X_te)}\n")

    # 5. Build & train
    model = build_model(SEQ_LEN, N_FEATURES)
    model.summary()

    callbacks = [
        cb.EarlyStopping(monitor="val_loss", patience=15,
                         restore_best_weights=True, verbose=1),
        cb.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                             patience=7, min_lr=1e-6, verbose=1),
        cb.ModelCheckpoint("burnout_lstm_model.h5", monitor="val_loss",
                           save_best_only=True, verbose=1),
    ]

    print("\n─── Training ─────────────────────────────────────────────")
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_v, y_v),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )

    # 6. Evaluate
    print("\n─── Test Results ─────────────────────────────────────────")
    results = model.evaluate(X_te, y_te, verbose=0)
    print(f"  MSE  : {results[0]:.5f}")
    print(f"  MAE  : {results[1]:.4f}")
    print(f"  RMSE : {results[2]:.4f}")
    y_pred = model.predict(X_te, verbose=0).flatten()
    to_level = lambda s: "Low" if s<0.35 else ("Moderate" if s<0.65 else "High")
    level_acc = np.mean([to_level(p)==to_level(t) for p,t in zip(y_pred, y_te)])
    print(f"  Level accuracy : {level_acc*100:.1f}%")
    print(f"  Best val_loss  : {min(history.history['val_loss']):.5f}")

    # 7. Save
    joblib.dump(scaler,       "lstm_scaler.pkl")
    joblib.dump(FEATURE_COLS, "feature_cols.pkl")
    joblib.dump(baseline,     "baseline_stats.pkl")
    with open("app_category_weights.json", "w") as f:
        json.dump(APP_CATEGORY_WEIGHTS, f, indent=2)

    print("\n─── Saved ────────────────────────────────────────────────")
    for fn in ["burnout_lstm_model.h5", "lstm_scaler.pkl",
               "feature_cols.pkl", "baseline_stats.pkl",
               "app_category_weights.json"]:
        if os.path.exists(fn):
            print(f"  ✓  {fn:<42} {os.path.getsize(fn)/1024:.1f} KB")

    print("\nDone! Copy all files above into your Backend/ folder.")
    return model, scaler, baseline


# ─── Entry point ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default=None)
    args, _unknown = parser.parse_known_args()
    run(args.data or DATA_PATH)
else:
    run(DATA_PATH)

TensorFlow 2.20.0  |  tf_keras 2.20.0
Generating synthetic data …
Saved synthetic_burnout_data.csv  (18000 rows)
Dataset: 18000 rows | 600 users

Baseline (mean ± std):
  screen_time_hours                 6.320 ± 2.194
  app_switches_per_hour             31.956 ± 15.628
  social_app_ratio                  0.341 ± 0.142
  work_app_ratio                    0.323 ± 0.140
  sleep_hours                       6.722 ± 1.099

Sequences: X=(17400, 2, 23)  y=(17400,)
Burnout  min=0.028  mean=0.388  max=0.872
Train 13920 | Val 1740 | Test 1740

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequence_input (InputLayer  [(None, 2, 23)]           0         
 )                                                               
                                                                 
 lstm (LSTM)                 (None, 2, 64)             22528     
                                                 

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


215/218 [============================>.] - ETA: 0s - loss: 0.0040 - mae: 0.0357 - rmse: 0.0477
Epoch 2: val_loss improved from 0.00421 to 0.00181, saving model to burnout_lstm_model.h5
218/218 [==============================] - 2s 7ms/step - loss: 0.0040 - mae: 0.0357 - rmse: 0.0476 - val_loss: 0.0018 - val_mae: 0.0195 - val_rmse: 0.0257 - lr: 0.0010
Epoch 3/150
216/218 [============================>.] - ETA: 0s - loss: 0.0024 - mae: 0.0298 - rmse: 0.0397
Epoch 3: val_loss improved from 0.00181 to 0.00109, saving model to burnout_lstm_model.h5
218/218 [==============================] - 2s 9ms/step - loss: 0.0024 - mae: 0.0298 - rmse: 0.0397 - val_loss: 0.0011 - val_mae: 0.0151 - val_rmse: 0.0220 - lr: 0.0010
Epoch 4/150
217/218 [============================>.] - ETA: 0s - loss: 0.0018 - mae: 0.0272 - rmse: 0.0368
Epoch 4: val_loss improved from 0.00109 to 0.00072, saving model to burnout_lstm_model.h5
218/218 [==============================] - 2s 10ms/step - loss: 0.0018 - mae: 0.0272 

In [ ]:
"""
Run this in Colab AFTER training to convert the model
to SavedModel format which works on any TF version.

Just paste this as a new cell and run it.
"""
import tensorflow as tf
import tf_keras as keras
import os

print(f"TF: {tf.__version__}")

# ── Register the custom layer before loading ──────────────────────────────────
class AttentionPool(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = keras.layers.Dense(1, activation="tanh")
    def call(self, inputs):
        scores = self.score_dense(inputs)
        scores = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(inputs * scores, axis=1)
    def get_config(self):
        return super().get_config()

# ── Load the .h5 you already trained ─────────────────────────────────────────
model = keras.models.load_model(
    "burnout_lstm_model.h5",
    custom_objects={"AttentionPool": AttentionPool}
)
model.summary()

# ── Save as SavedModel folder — version-independent ──────────────────────────
tf.saved_model.save(model, "burnout_lstm_savedmodel")
print("✓ Saved: burnout_lstm_savedmodel/")

# Verify it loads back correctly
loaded = tf.saved_model.load("burnout_lstm_savedmodel")
print("✓ Verified: model loads correctly")

print("""
Download the entire 'burnout_lstm_savedmodel/' folder from Colab
(right-click → Download in file browser) and place it in Backend/.

Your Backend/ should look like:
  Backend/
    burnout_lstm_savedmodel/   ← folder with saved_model.pb inside
    lstm_scaler.pkl
    feature_cols.pkl
    baseline_stats.pkl
    best_burnout_model.pt
    burnout_scaler.pkl
    main.py
    ...
""")

TF: 2.20.0
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequence_input (InputLayer  [(None, 2, 23)]           0         
 )                                                               
                                                                 
 lstm (LSTM)                 (None, 2, 64)             22528     
                                                                 
 dropout (Dropout)           (None, 2, 64)             0         
                                                                 
 lstm_1 (LSTM)               (None, 2, 32)             12416     
                                                                 
 dropout_1 (Dropout)         (None, 2, 32)             0         
                                                                 
 attention_pool (AttentionP  (None, 32)                33        
 ool)                                             

In [ ]:
import shutil, os
from google.colab import files

# Put all backend files into one zip
os.makedirs("backend_upload", exist_ok=True)
shutil.copytree("burnout_lstm_savedmodel", "backend_upload/burnout_lstm_savedmodel")
shutil.copy("lstm_scaler.pkl",    "backend_upload/")
shutil.copy("feature_cols.pkl",   "backend_upload/")
shutil.copy("baseline_stats.pkl", "backend_upload/")

shutil.make_archive("all_backend_files", "zip", "backend_upload")
files.download("all_backend_files.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>